# Start here: synthetic test of the lightweight causal model

This notebook runs without your work data. It generates **synthetic PDP-view opportunities**, fits the standalone `LightweightDRLearner` from `causal_dml.py`, and checks it against known causal effects. This is the lightweight categorical DR model with a **Tweedie log-link revenue nuisance** and bounded LightGBM final trees. No random forest is used.

In GitHub: **Code → Codespaces → Create codespace on main**. Wait for setup, open this file under `notebooks/`, choose the Python kernel, and click **Run All**. You do not clone manually or close the repository. The repository is already your working folder in Codespaces. For an existing Codespace, pull the latest changes first and rebuild the container if you want the new automatic setup.

All code here is test/demo code. Passing these checks establishes that the example executes and recovers a deliberately simple signal, not that production causal assumptions hold. Keep the seed and defaults unchanged for the smoke test; changed sample sizes, noise, versions or seeds may change the statistical recovery checks.

In [ ]:
# Locate the checkout whether Jupyter starts in the repository root or notebooks/.
from pathlib import Path
import sys
import subprocess
import importlib.util

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / 'causal_dml.py').exists()), None)
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook from the DML_CausalCustom checkout.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The committed Codespaces configuration installs these automatically.
# Fallback for an older Codespace or a local notebook: install into THIS kernel.
required = ['numpy', 'scipy', 'pandas', 'sklearn', 'lightgbm',
            'joblib', 'matplotlib', 'pytest', 'psutil', 'ipykernel']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    '-r', str(REPO_ROOT / 'requirements-notebooks.txt')], check=True)
print('Repository:', REPO_ROOT)
print('Python:', sys.executable)


In [ ]:
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from scipy.special import expit
from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display
from causal_dml import LightweightDRLearner, TweedieMeanRegressor, FinalStageConfig

SEED = 2026
rng = np.random.default_rng(SEED)
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'synthetic_smoke_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('LightGBM:', lgb.__version__)


## 1. Generate data with known causal truth

Each synthetic user has two PDP views. `industry` and `data_definition` describe the same two synthetic segments so the causal signal is intentionally easy to diagnose. The carousel helps one segment by **€4 per view** and harms the other by **€2 per view**. `intent_score` affects both carousel exposure and revenue, so it belongs in **W**. All features are generated before treatment; no outcome information enters X or W.

Conditional mean revenue is `mu0 = 8 + 2*segment + 1.5*intent_score`; `mu1 = mu0 + true_effect`. We draw nonnegative, zero-heavy outcomes from a compound Poisson–Gamma Tweedie model with power 1.5. This matches the nuisance family in this controlled example. Treatment assignment has good overlap, but is not randomized. All relevant confounding is observed **by construction**. The truth columns are for checking only and never enter the fit.

In [ ]:
N_USERS = 4_000
VIEWS_PER_USER = 2
user_id = np.repeat(np.arange(N_USERS), VIEWS_PER_USER)
segment = np.repeat(rng.integers(0, 2, N_USERS), VIEWS_PER_USER)
intent = rng.uniform(-1, 1, len(user_id))
true_effect = np.where(segment == 1, 4.0, -2.0)
mu0 = 8.0 + 2.0*segment + 1.5*intent
mu1 = mu0 + true_effect
propensity = expit(0.8*intent + 0.3*segment - 0.15)
exposed = rng.binomial(1, propensity)
observed_mean = np.where(exposed == 1, mu1, mu0)

# Exact compound Poisson-Gamma sampler for 1 < power < 2.
# E[Y|features,treatment] = mu; Var[Y|...] = dispersion * mu**power.
# A zero Poisson event count creates exactly zero revenue (no log1p needed).
def draw_tweedie(mu, *, power=1.5, dispersion=3.0, rng):
    mu = np.asarray(mu, dtype=float)
    rate = mu**(2-power) / (dispersion*(2-power))
    event_count = rng.poisson(rate)
    shape_per_event = (2-power)/(power-1)
    scale = dispersion*(power-1)*mu**(power-1)
    y = np.zeros(len(mu))
    positive = event_count > 0
    y[positive] = rng.gamma(event_count[positive]*shape_per_event, scale[positive])
    return y

synthetic = pd.DataFrame({
    'user_id': user_id,
    'industry': np.where(segment == 1, 'biotech', 'academic'),
    'data_definition': np.where(segment == 1, 'lab_water', 'solvents'),
    'intent_score': intent,
    'carousel': np.where(exposed == 1, 'carousel', 'no_carousel'),
    'revenue': draw_tweedie(observed_mean, rng=rng),
    'true_mu0': mu0, 'true_mu1': mu1, 'true_effect': true_effect,
    'true_propensity': propensity,
})
assert np.isfinite(synthetic['revenue']).all() and synthetic['revenue'].ge(0).all()
assert synthetic['revenue'].eq(0).any()
display(synthetic.head())
print(f'Rows: {len(synthetic):,}; users: {N_USERS:,}; zeros: {synthetic.revenue.eq(0).mean():.1%}')
display(synthetic.groupby(['industry','carousel']).agg(
    rows=('revenue','size'), observed_mean_eur=('revenue','mean')))
# Observed arm differences are NOT causal estimates when exposure is confounded.


## 2. Split users, then transform X and W

We reserve 25% of **users**, not rows, for independent testing. Both views from a user stay together. The encoder learns its category schema from training rows only. X is small and controls effect heterogeneity; W supplies observed confounders. IDs are split/cluster keys, never model features.

Revenue stays in original euros. The outcome nuisance handles the log link internally. `categories=['no_carousel','carousel']` explicitly fixes the reference and effect direction. There is one treatment-label vector, not a 500,000-product treatment matrix.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(synthetic, groups=synthetic['user_id']))
train = synthetic.iloc[train_idx].copy()
test = synthetic.iloc[test_idx].copy()
assert set(train.user_id).isdisjoint(set(test.user_id))

from sklearn.preprocessing import OneHotEncoder
X_COLUMNS = ['industry', 'data_definition']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32)
X_train = encoder.fit_transform(train[X_COLUMNS]).tocsr()
X_test = encoder.transform(test[X_COLUMNS]).tocsr()
W_train = train[['intent_score']].to_numpy(dtype=np.float32)
W_test = test[['intent_score']].to_numpy(dtype=np.float32)
Y_train = train['revenue'].to_numpy(dtype=float)  # No log1p and no cent/euro ambiguity.
Y_test = test['revenue'].to_numpy(dtype=float)
T_train = train['carousel'].to_numpy()
T_test = test['carousel'].to_numpy()
g_train = train['user_id'].to_numpy()
g_test = test['user_id'].to_numpy()
print('Train/test rows:', len(train), len(test))
print('X shapes:', X_train.shape, X_test.shape)


## 3. Create and fit the standalone model

Cross-fitting keeps each nuisance prediction out of the row's training fold and respects user groups. The Tweedie nuisance estimates nonnegative conditional revenue means with a log link. DR pseudo-outcomes are **signed**, so the final stage uses ordinary squared-error LightGBM trees. Few small trees, minimum leaf support, shrinkage and stochastic splits bound model complexity; these are not honest causal forests and do not provide forest-style pointwise inference.

In [ ]:
outcome_model = TweedieMeanRegressor(
    variance_power=1.5,
    model_params=dict(n_estimators=140, learning_rate=0.05,
                      num_leaves=7, min_child_samples=100, n_jobs=2,
                      random_state=SEED, verbosity=-1),
)
propensity_model = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.05, num_leaves=7,
    min_child_samples=100, reg_lambda=5, n_jobs=2,
    random_state=SEED, verbosity=-1,
)
model = LightweightDRLearner(
    categories=['no_carousel', 'carousel'],
    model_y=outcome_model, model_t=propensity_model,
    final_config=FinalStageConfig(n_estimators=100, num_leaves=7,
                                 max_depth=3, min_child_samples=200),
    cv=3, propensity_floor=0.01, batch_size=2_000,
    n_jobs=2, random_state=SEED,
)
started = time.perf_counter()
model.fit(Y_train, T_train, X=X_train, W=W_train, groups=g_train)
print(f'Fit finished in {time.perf_counter()-started:.2f} seconds')
display(pd.DataFrame(model.fold_audit_))
assert all(fold['shared_groups'] == 0 for fold in model.fold_audit_)


## 4. Test effects against the known truth

`effect(X,T0='no_carousel',T1='carousel')` returns expected incremental **euros per PDP view**. The reverse contrast must be its negative. For this binary setup, `const_marginal_effect(X)` has one column with the same contrast. We check exact API identities separately from a broad statistical recovery tolerance. The tolerance is a smoke-test target for this fixed simulation, not a confidence interval or an acceptable production error budget.

In [ ]:
effect = model.effect(X_test, T0='no_carousel', T1='carousel')
reverse = model.effect(X_test, T0='carousel', T1='no_carousel')
cme = model.const_marginal_effect(X_test)
assert effect.shape == (len(test),)
assert cme.shape == (len(test), 1)
assert np.isfinite(effect).all()
np.testing.assert_allclose(reverse, -effect, atol=1e-7)
np.testing.assert_allclose(cme[:, 0], effect, atol=1e-7)
np.testing.assert_allclose(model.effect(X_test, T0='carousel', T1='carousel'), 0, atol=1e-7)

comparison = test[['industry','data_definition','true_effect']].copy()
comparison['estimated_effect_eur'] = effect
cohorts = comparison.groupby(['industry','data_definition']).agg(
    test_rows=('true_effect','size'),
    true_effect_eur=('true_effect','mean'),
    estimated_effect_eur=('estimated_effect_eur','mean'),
)
cohorts['absolute_error_eur'] = abs(cohorts.estimated_effect_eur-cohorts.true_effect_eur)
display(cohorts)
assert (np.sign(cohorts.estimated_effect_eur) == np.sign(cohorts.true_effect_eur)).all()
assert cohorts.absolute_error_eur.max() < 1.5, 'Signal recovery failed: inspect versions/data/settings.'
cohorts[['true_effect_eur','estimated_effect_eur']].plot.bar(rot=0, figsize=(8,4))
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Carousel minus no carousel: euros per view')
plt.title('Synthetic truth versus held-out CATE predictions')
plt.tight_layout()
plt.show()
print('PASS: shapes, finite effects, contrast direction, API agreement, and signal recovery.')


## 5. Test a learned recommendation policy on held-out users

The policy shows the carousel when the **training-fitted** CATE prediction is positive. Its rule is fixed before using test outcomes. The comparator here is **no carousel for everyone**, not the observed current mix. No deployment costs are supplied, so the values are revenue, not profit.

The DR evaluator reports mean policy revenue, mean comparator revenue, incremental euros and `100 * incremental_euros / comparator_mean`. This percentage is a ratio of population means, not `exp(effect)-1` and not a mean of row percentages. The interval is for incremental euros and clusters repeated views by user. It is conditional on the learned policy/nuisances and does not address hidden confounding. No pointwise CATE interval is claimed. The synthetic conditional means let us compare against the exact expected value for this test population; we do not assert a 95% interval always contains it.

In [ ]:
policy = np.where(effect > 0, 'carousel', 'no_carousel')
value = model.evaluate_policy(
    Y_test, T_test, X=X_test, W=W_test, groups=g_test,
    policy=policy, baseline='no_carousel', min_propensity=0.05,
)
true_policy_mean = np.where(policy == 'carousel', test.true_mu1, test.true_mu0).mean()
true_baseline_mean = test.true_mu0.mean()
true_gain = true_policy_mean-true_baseline_mean
summary = pd.DataFrame({
    'quantity': ['Baseline mean EUR/view','Policy mean EUR/view',
                 'Incremental EUR/view','Percent change in mean revenue'],
    'DR_estimate': [value['baseline_mean_value'], value['policy_mean_value'],
                    value['mean_gain'], value['relative_gain_pct']],
    'synthetic_truth': [true_baseline_mean, true_policy_mean, true_gain,
                        100*true_gain/true_baseline_mean],
})
display(summary)
print('95% incremental euro interval:', value['ci_low'], value['ci_high'])
print('Overlap:', value['overlap_pass'], 'unsupported fraction:', value['unsupported_fraction'])
print('Policy / baseline ESS:', value['policy_ess'], value['baseline_ess'])
assert value['overlap_pass'], 'Inspect estimated propensities before interpreting the value.'
assert value['mean_gain'] > 0
assert abs(value['mean_gain']-true_gain) < 1.5
np.testing.assert_allclose(value['relative_gain_pct'],
                           100*value['mean_gain']/value['baseline_mean_value'])
print('PASS: held-out policy evaluation and euro/percentage consistency.')
# A deployment total would be incremental EUR/view times an explicitly chosen
# number of comparable future eligible views; generalization is an assumption.


## 6. Save and reload the small reusable model

The artifact includes the encoder and feature lists as well as the fitted causal estimator. The file size measures **serialized model size**, not peak training RAM. The notebook is designed to be small enough for a basic Codespace; it is not a production memory benchmark. Load only your own trusted joblib artifacts.

In [ ]:
artifact_path = OUTPUT_DIR / 'synthetic_causal_model.joblib'
joblib.dump({'model':model, 'encoder':encoder,
             'x_columns':X_COLUMNS, 'w_columns':['intent_score']}, artifact_path)
restored = joblib.load(artifact_path)
np.testing.assert_allclose(restored['model'].effect(X_test,
    T0='no_carousel', T1='carousel'), effect)
cohorts.to_csv(OUTPUT_DIR / 'effect_checks.csv')
summary.to_csv(OUTPUT_DIR / 'policy_checks.csv', index=False)
(OUTPUT_DIR / 'policy_diagnostics.json').write_text(json.dumps(value, indent=2))
print(f'Serialized artifact: {artifact_path.stat().st_size/1024**2:.3f} MiB')
print('Saved checks under:', OUTPUT_DIR)
print('PASS: serialization preserves predictions.')


## 7. Run the repository unit tests

Leave this enabled for your first run. The subprocess uses the active notebook's Python interpreter and runs the R learner, DR learner, pipeline and Tweedie tests. A nonzero exit code fails this cell instead of silently continuing. The model recovery checks above are additional notebook checks, separate from the unit suite.

In [ ]:
RUN_REPOSITORY_TESTS = True
if RUN_REPOSITORY_TESTS:
    result = subprocess.run([sys.executable, '-m', 'pytest', '-q'],
                            cwd=REPO_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
print('ALL ENABLED CHECKS PASSED. Synthetic smoke test complete.')


## Next: your actual data

Use `01_tweedie_revenue_pipeline.ipynb` for the full recommendation pipeline with real dataframe mapping, treatment definitions and cohort reports. This synthetic notebook directly exercises the reusable estimator API so you can isolate model behavior first.

If a cell fails, keep the traceback and rerun from the top after fixing it. Do not loosen the recovery tolerance to conceal a regression. Changing the simulation is an experiment, not the default reproducible check.

Save changes with **Ctrl+S** (or File → Save). Saving in Codespaces is not a GitHub commit: use Source Control to commit and sync any notebook/code changes you want in the repository. The notebook's generated data/artifacts live under ignored `outputs/` and are not pushed automatically. At the end, use **Codespaces: Stop Current Codespace** in the command palette, or stop it from https://github.com/codespaces. Closing the tab alone does not immediately stop compute. Resume the same Codespace later instead of creating a fresh one each time.

References: [Create a Codespace](https://docs.github.com/en/codespaces/developing-in-a-codespace/creating-a-codespace-for-a-repository), [stop/restart](https://docs.github.com/en/codespaces/developing-in-a-codespace/stopping-and-starting-a-codespace), and the repository's `METHODS_V2.md` and `REVENUE_REPORTING.md`.